# Prepare the official testing set

So far every score comes from the **validation** set, which the models also used for early stopping, threshold choice and tuning. The official `UNSW_NB15_testing-set.csv` has never been touched, so it gives the honest final numbers.

This notebook processes the testing set with **exactly the same steps and the same fitted scaler** as the training data (it reads the settings from `processed/meta.json`), then saves it as `processed/unsw_nb15_test.npz`. It also runs two checks a careful reviewer will ask about:
1. **Drift check:** do the scaled test features look like the training features?
2. **Leakage check:** how many test rows are identical to training rows?

Run it after the preprocessing notebook. Put `UNSW_NB15_testing-set.csv` next to `processed/`.

## 1. Imports and config

In [1]:
import json
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from IPython.display import display

TEST_CSV = "UNSW_NB15_testing-set.csv"
PROCESSED_DIR = "processed"

## 2. Load the saved artifacts and mirror the training settings

In [2]:
proc = Path(PROCESSED_DIR)
scaler = joblib.load(proc / "scaler.joblib")
meta = json.loads((proc / "meta.json").read_text())
feature_names, scale_cols, class_names = meta["feature_names"], meta["scale_cols"], meta["class_names"]

cfg = meta.get("config", {})
USE_EXTRA = cfg.get("add_extra_features", "rate" in feature_names)
USE_STATE = cfg.get("use_state", any(c.startswith("state_") for c in feature_names))
RECOMPUTE_DERIVED = cfg.get("recompute_derived", True)
MERGED = "Exploit-like" in class_names
print(f"extra features: {USE_EXTRA}   state: {USE_STATE}   recompute derived: {RECOMPUTE_DERIVED}   merged classes: {MERGED}")
print("classes:", class_names)

extra features: False   state: True   recompute derived: True   merged classes: False
classes: ['Normal', 'DoS', 'Reconnaissance', 'Fuzzers', 'Exploits', 'Generic', 'Other']


## 3. The same cleaning and feature code as the preprocessing notebook

This is a copy of the functions from the preprocessing notebook. If you ever change that notebook, change this cell too.

In [3]:
REQUIRED_COLUMNS = [
    "dur", "proto", "service", "state", "sbytes", "dbytes", "spkts", "dpkts",
    "sload", "dload", "smean", "dmean", "is_sm_ips_ports",
    "ct_srv_src", "ct_srv_dst", "ct_dst_ltm", "ct_src_ltm",
    "ct_src_dport_ltm", "ct_dst_sport_ltm", "ct_dst_src_ltm", "attack_cat", "label",
]
CT_COLS = ["ct_srv_src", "ct_srv_dst", "ct_dst_ltm", "ct_src_ltm",
           "ct_src_dport_ltm", "ct_dst_sport_ltm", "ct_dst_src_ltm"]
BASE_NUMERIC_COLS = ["dur", "sbytes", "dbytes", "spkts", "dpkts",
                     "sload", "dload", "smean", "dmean", "is_sm_ips_ports", *CT_COLS]
EXTRA_COLS = ["rate", "sinpkt_approx", "dinpkt_approx", "byte_ratio", "pkt_ratio", "dur_zero"]
NUMERIC_COLS = BASE_NUMERIC_COLS + (EXTRA_COLS if USE_EXTRA else [])
LOG_COLS = ["dur", "sbytes", "dbytes", "spkts", "dpkts", "sload", "dload", "smean", "dmean",
            "rate", "sinpkt_approx", "dinpkt_approx"]

PROTO_CATS = ["tcp", "udp", "icmp", "other"]
SERVICE_CATS = ["-", "dns", "http", "smtp", "ftp", "ftp-data", "ssh",
                "pop3", "ssl", "irc", "dhcp", "snmp", "radius", "other"]
STATE_GROUPS = ["ok", "no_reply", "reset", "other"]
ARGUS_STATE_MAP = {"FIN": "ok", "CLO": "ok", "CON": "ok", "ACC": "ok",
                   "INT": "no_reply", "REQ": "no_reply", "RST": "reset"}
CAT_SPEC = {"proto": PROTO_CATS, "service": SERVICE_CATS, "state": STATE_GROUPS}

if MERGED:
    ATTACK_TO_CLASS = {"Reconnaissance": "Reconnaissance", "Fuzzers": "Fuzzers", "Generic": "Generic"}
    OTHER_ATTACK_CLASS = "Exploit-like"
else:
    ATTACK_TO_CLASS = {"DoS": "DoS", "Reconnaissance": "Reconnaissance", "Fuzzers": "Fuzzers",
                       "Exploits": "Exploits", "Generic": "Generic"}
    OTHER_ATTACK_CLASS = "Other"


def group_proto(s):
    s = s.fillna("").astype(str).str.strip().str.lower()
    return s.where(s.isin(PROTO_CATS[:-1]), "other")


def group_service(s):
    s = s.fillna("-").astype(str).str.strip().str.lower().replace("", "-")
    return s.where(s.isin(SERVICE_CATS[:-1]), "other")


def group_state(s, state_map):
    s = s.fillna("").astype(str).str.strip().str.upper()
    return s.map(state_map).fillna("other")


def safe_div(a, b):
    a = np.asarray(a, dtype="float64"); b = np.asarray(b, dtype="float64")
    out = np.zeros_like(a)
    np.divide(a, b, out=out, where=b > 0)
    return out


def clean_and_engineer(df):
    df = df.copy()
    for c in BASE_NUMERIC_COLS:
        df[c] = pd.to_numeric(df[c], errors="coerce").fillna(0)
    df["label"] = pd.to_numeric(df["label"], errors="coerce").fillna(0).astype(int)
    df["proto"] = group_proto(df["proto"])
    df["service"] = group_service(df["service"])
    df["state"] = group_state(df["state"], ARGUS_STATE_MAP)
    if RECOMPUTE_DERIVED:
        df["sload"] = safe_div(df["sbytes"] * 8, df["dur"])
        df["dload"] = safe_div(df["dbytes"] * 8, df["dur"])
        df["smean"] = safe_div(df["sbytes"], df["spkts"])
        df["dmean"] = safe_div(df["dbytes"], df["dpkts"])
    df["is_sm_ips_ports"] = (df["is_sm_ips_ports"] > 0).astype(int)
    for c in CT_COLS:
        df[c] = df[c].clip(lower=0)
    if USE_EXTRA:
        df["rate"] = safe_div(df["spkts"] + df["dpkts"] - 1, df["dur"])
        df["sinpkt_approx"] = safe_div(df["dur"] * 1000, df["spkts"] - 1)
        df["dinpkt_approx"] = safe_div(df["dur"] * 1000, df["dpkts"] - 1)
        df["byte_ratio"] = np.log1p(df["sbytes"]) - np.log1p(df["dbytes"])
        df["pkt_ratio"] = np.log1p(df["spkts"]) - np.log1p(df["dpkts"])
        df["dur_zero"] = (df["dur"] <= 0).astype(int)
    return df


def build_targets(df):
    cat = (df["attack_cat"].fillna("Normal").astype(str).str.strip()
           .replace({"Backdoor": "Backdoors", "": "Normal"}))
    group = cat.map(ATTACK_TO_CLASS).fillna(OTHER_ATTACK_CLASS)
    group = group.mask(df["label"] == 0, "Normal")
    df = df.copy()
    df["attack_group"] = group
    df["y_multi"] = group.map({c: i for i, c in enumerate(class_names)}).astype(int)
    return df


def build_model_frame(df):
    num = df[NUMERIC_COLS].astype("float64").copy()
    for c in LOG_COLS:
        if c in num.columns:
            num[c] = np.log1p(num[c].clip(lower=0))
    parts = [num]
    for col, cats in CAT_SPEC.items():
        if col == "state" and not USE_STATE:
            continue
        cat = pd.Series(pd.Categorical(df[col], categories=cats), index=df.index)
        parts.append(pd.get_dummies(cat, prefix=col).astype("float64"))
    return pd.concat(parts, axis=1)

## 4. Process the testing set

In [4]:
raw = pd.read_csv(TEST_CSV, low_memory=False, encoding="latin-1")
raw.columns = [c.strip().lower() for c in raw.columns]
missing = [c for c in REQUIRED_COLUMNS if c not in raw.columns]
if missing:
    raise ValueError(f"Missing expected columns: {missing}\nColumns found: {list(raw.columns)}")
print(f"loaded {len(raw):,} rows")

df = build_targets(clean_and_engineer(raw))
frame = build_model_frame(df).reindex(columns=feature_names, fill_value=0.0).copy()
frame[scale_cols] = scaler.transform(frame[scale_cols])          # the scaler fitted on the TRAIN set only

X_test = frame.to_numpy(dtype="float32")
y_bin_test = df["label"].to_numpy().astype("int8")
y_multi_test = df["y_multi"].to_numpy().astype("int8")
print("X_test:", X_test.shape)

loaded 82,332 rows
X_test: (82332, 39)


## 5. Class balance: training vs testing

In [5]:
data = np.load(proc / "unsw_nb15_processed.npz")
X_train = data["X_train"]
train_share = pd.Series(data["y_multi_train"]).map(dict(enumerate(class_names))).value_counts(normalize=True)
test_share = pd.Series(y_multi_test).map(dict(enumerate(class_names))).value_counts(normalize=True)
balance = pd.DataFrame({"train %": 100 * train_share, "test %": 100 * test_share}).reindex(class_names)
display(balance.round(2))
print(f"attack share  train: {100 * data['y_bin_train'].mean():.1f}%   test: {100 * y_bin_test.mean():.1f}%")

,train %,test %
Normal,31.94,44.94
DoS,6.99,4.97
Reconnaissance,5.98,4.25
Fuzzers,10.37,7.36
Exploits,19.05,13.52
Generic,22.81,22.92
Other,2.86,2.04


attack share  train: 68.1%   test: 55.1%


## 6. Drift check

The training features were standardised, so they have mean 0 and standard deviation 1. Large shifts in the test set mean the two sets differ (a different attack mix shifts some features, which is expected).

In [6]:
t_mean, t_std = X_train.mean(0), X_train.std(0)
shift = (X_test.mean(0) - t_mean) / np.maximum(t_std, 1e-6)
drift = pd.DataFrame({"feature": feature_names, "test_mean": X_test.mean(0), "test_std": X_test.std(0),
                      "mean_shift_in_std": shift}).reindex(np.argsort(-np.abs(shift)))
display(drift.head(10).round(3))

,feature,test_mean,test_std,mean_shift_in_std
20,proto_other,0.119,0.324,-0.142
17,proto_tcp,0.523,0.499,0.122
16,ct_dst_src_ltm,-0.118,1.045,-0.118
5,sload,-0.108,1.023,-0.108
15,ct_dst_sport_ltm,-0.093,1.020,-0.093
21,service_-,0.573,0.495,0.079
35,state_ok,0.563,0.496,0.072
36,state_no_reply,0.437,0.496,-0.071
9,is_sm_ips_ports,0.011,0.105,0.070
13,ct_src_ltm,-0.061,1.028,-0.061


## 7. Leakage check

Counts test rows whose feature vector is identical to some training row. A high share does not break the evaluation, but it makes the test score optimistic, so report the number.

In [7]:
h_train = pd.util.hash_pandas_object(pd.DataFrame(np.round(X_train, 4)), index=False).to_numpy()
h_test = pd.util.hash_pandas_object(pd.DataFrame(np.round(X_test, 4)), index=False).to_numpy()
overlap = np.isin(h_test, h_train)
print(f"test rows identical to a training row: {overlap.sum():,} of {len(overlap):,} ({100 * overlap.mean():.1f}%)")

by_class = (pd.DataFrame({"class": pd.Series(y_multi_test).map(dict(enumerate(class_names))), "identical": overlap})
            .groupby("class")["identical"].mean().mul(100).round(1).reindex(class_names))
display(by_class.to_frame("% identical to a training row"))

test rows identical to a training row: 7,535 of 82,332 (9.2%)


,% identical to a training row
class,
Normal,3.8
DoS,43.0
Reconnaissance,30.6
Fuzzers,3.7
Exploits,14.2
Generic,7.2
Other,8.0


## 8. Save

In [8]:
np.savez_compressed(proc / "unsw_nb15_test.npz", X_test=X_test, y_bin_test=y_bin_test, y_multi_test=y_multi_test,
                    identical_to_train=overlap)
print("saved", proc / "unsw_nb15_test.npz")

saved processed\unsw_nb15_test.npz
